In [2]:
# Locate the project before importing source.
from pathlib import Path
import sys

cwd = Path.cwd().resolve()
project_root = next(
    (path for path in (cwd, *cwd.parents)
     if (path / "source" / "model.py").is_file()),
    None,
)
if project_root is None:
    raise RuntimeError("Open this notebook from inside the ImageProcessing_Lab02 project.")

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from source.config import CONFIG
from source.model import MLP, initialize_model

model = MLP(
    input_dim=CONFIG.input_dim,
    hidden_dims=CONFIG.hidden_dims,
    activation=CONFIG.activation
)

initialize_model(
    model,
    CONFIG.initialization
)

print(model)

MLP(
  (layers): Sequential(
    (0): Linear(in_features=12288, out_features=128, bias=True)
    (1): CustomActivation()
    (2): Linear(in_features=128, out_features=64, bias=True)
    (3): CustomActivation()
    (4): Linear(in_features=64, out_features=1, bias=True)
  )
)


In [3]:
import torch

x = torch.randn(1, 3, 64, 64)

output = model(x)

assert output.shape == (1, 1)
assert torch.isfinite(output).all()

print("Single image forward: PASS")

Single image forward: PASS


In [4]:
x = torch.randn(32, 3, 64, 64)

output = model(x)

assert output.shape == (32, 1)
assert torch.isfinite(output).all()

print("Batch forward: PASS")

Batch forward: PASS


In [5]:
total_params = sum(
    p.numel()
    for p in model.parameters()
)

trainable_params = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)

print("Total parameters:", total_params)
print("Trainable parameters:", trainable_params)

assert total_params == 1581313
assert trainable_params == 1581313

print("Parameter registration: PASS")

Total parameters: 1581313
Trainable parameters: 1581313
Parameter registration: PASS


In [6]:
x = torch.randn(32, 3, 64, 64)

targets = torch.randint(
    0, 2, (32, 1)
).float()

criterion = torch.nn.BCEWithLogitsLoss()

model.zero_grad()

logits = model(x)

loss = criterion(logits, targets)

loss.backward()

for name, param in model.named_parameters():

    assert param.grad is not None
    assert torch.isfinite(param.grad).all()

    print(
        name,
        "Gradient shape:",
        param.grad.shape
    )

print("Backward propagation: PASS")

layers.0.weight Gradient shape: torch.Size([128, 12288])
layers.0.bias Gradient shape: torch.Size([128])
layers.2.weight Gradient shape: torch.Size([64, 128])
layers.2.bias Gradient shape: torch.Size([64])
layers.4.weight Gradient shape: torch.Size([1, 64])
layers.4.bias Gradient shape: torch.Size([1])
Backward propagation: PASS


In [7]:
from source.dataset import build_loaders
from source.augmentation import build_augmentation

loaders = build_loaders(
    CONFIG,
    train_transform=build_augmentation(CONFIG)
)

images, targets = next(iter(loaders["train"]))

logits = model(images)

assert logits.shape == targets.shape

loss = criterion(logits, targets)

assert torch.isfinite(loss)

print("Images:", images.shape)
print("Targets:", targets.shape)
print("Logits:", logits.shape)
print("Loss:", loss.item())

print("End-to-end integration: PASS")

Images: torch.Size([32, 3, 64, 64])
Targets: torch.Size([32, 1])
Logits: torch.Size([32, 1])
Loss: 0.7368895411491394
End-to-end integration: PASS


In [8]:
import torch
import random

torch.manual_seed(CONFIG.seed)
random.seed(CONFIG.seed)

test_model = MLP(
    input_dim=CONFIG.input_dim,
    hidden_dims=CONFIG.hidden_dims,
    activation=CONFIG.activation
)

initialize_model(
    test_model,
    CONFIG.initialization
)

criterion = torch.nn.BCEWithLogitsLoss()

optimizer = torch.optim.Adam(
    test_model.parameters(),
    lr=CONFIG.learning_rate
)

# Keep the same batch throughout training
images, targets = next(iter(loaders["train"]))

loss_history = []

test_model.train()

for step in range(100):

    # Reset gradients
    optimizer.zero_grad()

    # Forward
    logits = test_model(images)

    # Calculate loss
    loss = criterion(logits, targets)

    # Backward
    loss.backward()

    # Update weights
    optimizer.step()

    loss_history.append(loss.item())

    if (step + 1) % 10 == 0:
        print(
            f"Step {step + 1:03d} | "
            f"Loss: {loss.item():.6f}"
        )

Step 010 | Loss: 0.976464
Step 020 | Loss: 0.440679
Step 030 | Loss: 0.187307
Step 040 | Loss: 0.080639
Step 050 | Loss: 0.039745
Step 060 | Loss: 0.031513
Step 070 | Loss: 0.023444
Step 080 | Loss: 0.020039
Step 090 | Loss: 0.017032
Step 100 | Loss: 0.014810


In [9]:
print("Initial loss:", loss_history[0])
print("Final loss:", loss_history[-1])

assert all(
    torch.isfinite(torch.tensor(loss_history))
)

assert loss_history[-1] < loss_history[0]

print("Single Batch Learning: PASS")

Initial loss: 1.5608587265014648
Final loss: 0.014809851534664631
Single Batch Learning: PASS


In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(10, 5))

ax.plot(
    range(1, len(loss_history) + 1),
    loss_history
)

ax.set_title("Single Batch Overfitting Test")
ax.set_xlabel("Optimization Step")
ax.set_ylabel("Binary Cross Entropy Loss")
ax.grid(alpha=0.3)

fig.tight_layout()

output_path = (
    project_root / "results/figures/single_batch_overfit.png"
)

output_path.parent.mkdir(
    parents=True,
    exist_ok=True
)

fig.savefig(
    output_path,
    dpi=200,
    bbox_inches="tight"
)

plt.show()